In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
credits = pd.read_csv('tmdb_5000_credits.csv')
movies = pd.read_csv('tmdb_5000_movies.csv')
print(movies.columns)

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')


In [3]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [4]:
movies = movies.merge(credits, left_on='title', right_on='title')
#Menggabungkan Data: Data digabungkan berdasarkan kolom title, sehingga semua informasi terkait film 
#(seperti pemain, kru, genre, dan ringkasan) berada dalam satu DataFrame.

In [5]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [6]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'vote_count']]


In [8]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,vote_count
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,11800


In [9]:
#Preprocessing Data
#Mengkonversi string JSON-like ke list nama / list python
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [10]:
movies['genres'] = movies['genres'].apply(convert)

In [11]:
#Hasilnya, kolom genres akan memiliki daftar nama genre, misalnya:
movies['genres']

0       [Action, Adventure, Fantasy, Science Fiction]
1                        [Adventure, Fantasy, Action]
2                          [Action, Adventure, Crime]
3                    [Action, Crime, Drama, Thriller]
4                [Action, Adventure, Science Fiction]
                            ...                      
4804                        [Action, Crime, Thriller]
4805                                [Comedy, Romance]
4806               [Comedy, Drama, Romance, TV Movie]
4807                                               []
4808                                    [Documentary]
Name: genres, Length: 4809, dtype: object

In [12]:
movies['keywords'] = movies['keywords'].apply(convert)

In [13]:
movies['keywords']

0       [culture clash, future, space war, space colon...
1       [ocean, drug abuse, exotic island, east india ...
2       [spy, based on novel, secret agent, sequel, mi...
3       [dc comics, crime fighter, terrorist, secret i...
4       [based on novel, mars, medallion, space travel...
                              ...                        
4804    [united states–mexico barrier, legs, arms, pap...
4805                                                   []
4806    [date, love at first sight, narration, investi...
4807                                                   []
4808            [obsession, camcorder, crush, dream girl]
Name: keywords, Length: 4809, dtype: object

In [14]:
movies['cast'] = movies['cast'].apply(lambda x: [i['name'] for i in ast.literal_eval(x)[:3]])  # Only top 3 actors

In [15]:
movies['crew'] = movies['crew'].apply(lambda x: [i['name'] for i in ast.literal_eval(x) if i['job'] == 'Director']) # Only Directors

In [16]:
movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
#Membuat kolom baru tags yang menggabungkan nilai dari beberapa kolom lainnya: genres, keywords, cast, dan crew.

In [17]:
movies['tags']

0       [Action, Adventure, Fantasy, Science Fiction, ...
1       [Adventure, Fantasy, Action, ocean, drug abuse...
2       [Action, Adventure, Crime, spy, based on novel...
3       [Action, Crime, Drama, Thriller, dc comics, cr...
4       [Action, Adventure, Science Fiction, based on ...
                              ...                        
4804    [Action, Crime, Thriller, united states–mexico...
4805    [Comedy, Romance, Edward Burns, Kerry Bishé, M...
4806    [Comedy, Drama, Romance, TV Movie, date, love ...
4807    [Daniel Henney, Eliza Coupe, Bill Paxton, Dani...
4808    [Documentary, obsession, camcorder, crush, dre...
Name: tags, Length: 4809, dtype: object

In [18]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))
#Menggabungkan elemen-elemen dalam kolom tags yang merupakan list (daftar) menjadi satu string, di mana setiap elemen dipisahkan dengan spasi.

In [19]:
movies['tags']

0       Action Adventure Fantasy Science Fiction cultu...
1       Adventure Fantasy Action ocean drug abuse exot...
2       Action Adventure Crime spy based on novel secr...
3       Action Crime Drama Thriller dc comics crime fi...
4       Action Adventure Science Fiction based on nove...
                              ...                        
4804    Action Crime Thriller united states–mexico bar...
4805    Comedy Romance Edward Burns Kerry Bishé Marsha...
4806    Comedy Drama Romance TV Movie date love at fir...
4807    Daniel Henney Eliza Coupe Bill Paxton Daniel Hsia
4808    Documentary obsession camcorder crush dream gi...
Name: tags, Length: 4809, dtype: object

In [20]:
movies = movies[['movie_id', 'title', 'overview', 'tags', 'vote_average', 'vote_count']]
#Kode ini digunakan untuk memilih hanya kolom-kolom tertentu dari DataFrame movies dan menyimpannya kembali dalam variabel movies. 
#Dengan kata lain, Anda hanya ingin bekerja dengan kolom movie_id, title, overview, tags, vote_average, dan vote_count dan mengabaikan kolom lainnya.

In [21]:
movies['tags'] = movies['tags'].apply(lambda x: x.lower())
#Kode ini digunakan untuk mengubah seluruh teks dalam kolom tags menjadi huruf kecil (lowercase).

In [22]:
def weighted_rating(x, m, C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + m) * R) + (m / (m + v) * C)


In [39]:
#Memperhitungkan jumlah vote (vote count) dan rating rata-rata (vote average) dari sebuah item (film, produk, dll.), dengan menggunakan nilai rating global (C) dan threshold minimum vote (m).
#Bagian pertama ((v / (v + m)) * R) memberikan lebih banyak bobot pada rating film jika jumlah vote v lebih tinggi dari m. Jika v lebih besar, 
#Artinya film tersebut memiliki lebih banyak kredibilitas, jadi ratingnya dihitung dengan lebih banyak bobot dari rating rata-rata film itu sendiri.
#Bagian kedua ((m / (m + v)) * C) memberikan bobot pada rating rata-rata global (C) ketika jumlah vote v kecil. Jika jumlah vote lebih kecil dari m, 
#Artinya film tersebut dianggap kurang kredibel, jadi lebih banyak bobot diberikan pada rating global (C), bukan rating film itu sendiri.

In [23]:
C = movies['vote_average'].mean()
m = movies['vote_count'].quantile(0.90)
high_rated_movies = movies.copy().loc[movies['vote_count'] >= m]
high_rated_movies['score'] = high_rated_movies.apply(lambda x: weighted_rating(x, m, C), axis=1)
high_rated_movies = high_rated_movies.sort_values('score', ascending=False)

In [24]:
def get_high_rated_movies(high_rated_movies):
    return high_rated_movies[['title', 'score']].head(10)
#Menghitung rating berbobot (weighted rating) untuk film yang memiliki jumlah vote lebih tinggi dari kuantil ke-90 (m),
#dan mengurutkan film-film tersebut berdasarkan nilai score yang dihitung dengan menggunakan fungsi weighted_rating


In [28]:
movies.head()

,movie_id,title,overview,tags,vote_average,vote_count
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",action adventure fantasy science fiction cultu...,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",adventure fantasy action ocean drug abuse exot...,6.9,4500
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,action adventure crime spy based on novel secr...,6.3,4466
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,action crime drama thriller dc comics crime fi...,7.6,9106
4,49529,John Carter,"John Carter is a war-weary, former military ca...",action adventure science fiction based on nove...,6.1,2124


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])
#Mengubah teks dari kolom tags menjadi matriks (TF-IDF matrix) fitur numerik yang dapat digunakan untuk perhitungan kemiripan.
#Setiap film diwakili oleh sebuah vektor yang menunjukkan pentingnya kata-kata dalam deskripsi film tersebut.)

In [26]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
#Cosine Similarity digunakan untuk mengukur kemiripan antar film berdasarkan vektor TF-IDF mereka. 
#Hasil dari cosine_similarity adalah sebuah matriks kemiripan di mana setiap elemen menunjukkan kemiripan antara dua film.
#Cosine similarity mengukur kemiripan arah antara dua vektor: nilai 1 berarti vektor tersebut identik (film sangat mirip), sedangkan nilai 0 berarti vektor tersebut tidak mirip sama sekali.

In [33]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = movies[movies['title'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:21]  # Get top 20 similar movies
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices]



#Mencari film yang dipilih berdasarkan judul.
#Menghitung kemiripan antara film yang dipilih dan film lain menggunakan Cosine Similarity.
#Mengurutkan film berdasarkan kemiripan tertinggi.
#Mengembalikan 20 film teratas yang paling mirip dengan film yang dipilih.

In [30]:
print(get_recommendations('The Dark Knight Rises'))

(                title  movie_id
65    The Dark Knight       155
119     Batman Begins       272
1360           Batman       268
210    Batman & Robin       415
428    Batman Returns       364
1361           Batman      2661
1197     The Prestige      1124
303          Catwoman       314, [0.4467781458465152, 0.38231390205367316, 0.23589894201430386, 0.22691069656853652, 0.21615683427890134, 0.21510376838309184, 0.19937527001752586, 0.17515202302135865])


In [32]:
import pickle
with open('movie_data.pkl', 'wb') as file:
    pickle.dump((movies, cosine_sim), file)
#Menyimpan objek Python ke dalam sebuah file menggunakan pickle (Modul bawaan di Python yang digunakan untuk serialisasi dan deserialisasi objek),
#yang memungkinkan Anda untuk menyimpan dan memuat data Python secara efisien.